<a href="https://colab.research.google.com/github/Charlesnorris509/Kaggle/blob/main/threat_detection_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Threat Detection using Computer Vision

This notebook demonstrates a system for identifying potential threats in military environments using computer vision. It utilizes Python, OpenCV, and TensorFlow to process video input (simulating a live camera feed), detect objects, classify them, and flag potential threats.

**Objectives:**

*   Set up the necessary Python environment with OpenCV and TensorFlow.
*   Preprocess video input.
*   Load a pre-trained object detection model.
*   Detect and classify objects in video frames.
*   Draw bounding boxes and labels, highlighting potential threats.
*   Display processed frames in real-time (simulated).
*   Generate a video output demonstrating the detection.
*   Ensure compatibility with Google Colab.
*   Provide clear explanations for each step.

## 1. Environment Setup

This section installs the necessary libraries. If running in Google Colab, these commands will set up the environment. If running locally, ensure you have these libraries installed in your Python environment.

In [1]:
# Install necessary libraries
!pip install opencv-python-headless tensorflow numpy requests
# Note: Using opencv-python-headless to avoid potential display issues in non-GUI environments like Colab/servers.
# If you need GUI features locally, you might install opencv-python instead.

In [2]:
# Import libraries
import cv2
import tensorflow as tf
import numpy as np
import os
import time
import requests # To download sample video

# Try importing Colab specific libraries, handle error if not in Colab
try:
    from google.colab.patches import cv2_imshow
    from IPython.display import HTML, display
    from base64 import b64encode
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    # Define placeholders if not in Colab to avoid NameErrors later
    def cv2_imshow(img):
        print("cv2_imshow is only available in Google Colab. Cannot display image.")
    def HTML(html_string):
        print("HTML display is only available in Google Colab.")
        return None
    def display(obj):
        print("Display function is only available in Google Colab.")
    def b64encode(data):
        import base64
        return base64.b64encode(data)
    print("Not running in Google Colab. cv2_imshow and HTML video display will not be used.")

## 2. Configuration and Setup

Define constants, paths, and specify the input video file. **Update `VIDEO_FILENAME` if your video is in a different location.**

In [3]:
# Configuration
# VIDEO_URL = "https://storage.googleapis.com/manus-sandbox-public-assets/sample_military_patrol.mp4" # Old URL - Not working

# --- IMPORTANT: SET YOUR VIDEO FILE PATH HERE ---
# If running in Colab, upload your video and update the path below.
# Example for Colab after uploading: VIDEO_FILENAME = "/content/Simply Ruthless.mp4"
VIDEO_FILENAME = "/content/SimplyRuthless.mp4" # Path in the sandbox environment
# --- End of Path Setting ---

OUTPUT_VIDEO_FILENAME = "output_threat_detection.mp4"
MODEL_NAME = "ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8" # Example TF Hub model

# Threat classes (redefined later based on model)
THREAT_CLASSES = ["person"]
CONFIDENCE_THRESHOLD = 0.5 # Minimum confidence score to consider a detection

# Check if input video exists
if not os.path.exists(VIDEO_FILENAME):
    print(f"ERROR: Input video file not found at {VIDEO_FILENAME}")
    print("Please ensure the video file exists at the specified path or update the VIDEO_FILENAME variable.")
    # Optionally, raise an error to stop execution if file not found
    # raise FileNotFoundError(f"Input video file not found: {VIDEO_FILENAME}")
else:
    print(f"Using input video: {VIDEO_FILENAME}")

print(f"Using TensorFlow Hub model: {MODEL_NAME}")
# Model loading will happen in the next step using hub.load()

Using input video: /content/SimplyRuthless.mp4
Using TensorFlow Hub model: ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8


## 3. Load Pre-trained Model

Load the selected object detection model from TensorFlow Hub. This model is pre-trained on the COCO dataset and can detect a variety of common objects.
We also define a helper function to get class names from IDs, adjusting the threat classes based on COCO limitations.

In [4]:
print("Loading model from TensorFlow Hub... This might take a moment.")
# Load the model from TensorFlow Hub
# Using hub.load directly handles download and caching
try:
    hub_model = tf.saved_model.load(f"https://tfhub.dev/tensorflow/{MODEL_NAME}/1")
    print("Model loaded successfully.")
    # Get the concrete function for inference
    detector = hub_model.signatures["serving_default"]
except Exception as e:
    print(f"Error loading model: {e}")
    # Handle error appropriately, e.g., raise e

# Helper function to map class IDs to names (COCO dataset)
def get_class_name(class_id):
    # COCO dataset labels (condensed)
    coco_labels = {
        1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'airplane',
        6: 'bus', 7: 'train', 8: 'truck', 9: 'boat', 10: 'traffic light',
        11: 'fire hydrant', 13: 'stop sign', 14: 'parking meter', 15: 'bench',
        16: 'bird', 17: 'cat', 18: 'dog', 19: 'horse', 20: 'sheep',
        21: 'cow', 22: 'elephant', 23: 'bear', 24: 'zebra', 25: 'giraffe',
        27: 'backpack', 28: 'umbrella', 31: 'handbag', 32: 'tie',
        33: 'suitcase', 34: 'frisbee', 35: 'skis', 36: 'snowboard',
        37: 'sports ball', 38: 'kite', 39: 'baseball bat', 40: 'baseball glove',
        41: 'skateboard', 42: 'surfboard', 43: 'tennis racket', 44: 'bottle',
        46: 'wine glass', 47: 'cup', 48: 'fork', 49: 'knife', 50: 'spoon',
        51: 'bowl', 52: 'banana', 53: 'apple', 54: 'sandwich', 55: 'orange',
        56: 'broccoli', 57: 'carrot', 58: 'hot dog', 59: 'pizza', 60: 'donut',
        61: 'cake', 62: 'chair', 63: 'couch', 64: 'potted plant', 65: 'bed',
        67: 'dining table', 70: 'toilet', 72: 'tv', 73: 'laptop', 74: 'mouse',
        75: 'remote', 76: 'keyboard', 77: 'cell phone', 78: 'microwave',
        79: 'oven', 80: 'toaster', 81: 'sink', 82: 'refrigerator', 84: 'book',
        85: 'clock', 86: 'vase', 87: 'scissors', 88: 'teddy bear',
        89: 'hair drier', 90: 'toothbrush'
        # Note: COCO doesn't have specific 'rifle', 'tank' etc.
    }
    # Update THREAT_CLASSES based on available COCO labels
    # This global modification is kept to match previous logic, but isn't ideal
    global THREAT_CLASSES
    THREAT_CLASSES = ["person"] # Re-defining based on COCO limitations

    return coco_labels.get(int(class_id), 'N/A')

# Call once to ensure THREAT_CLASSES is updated before processing starts
get_class_name(1) # Pass a dummy ID to trigger the global update
print(f"Threat classes considered (based on COCO): {THREAT_CLASSES}")

Loading model from TensorFlow Hub... This might take a moment.
Error loading model: File system scheme 'https' not implemented (file: 'https://tfhub.dev/tensorflow/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/1/saved_model.pb')
Threat classes considered (based on COCO): ['person']


## 4. Video Processing and Threat Detection

This section processes the input video frame by frame. For each frame, it performs object detection, identifies potential threats based on the defined classes and confidence threshold, and annotates the frame with bounding boxes and labels.

In [7]:
# Check if video file exists before trying to open
if not os.path.exists(VIDEO_FILENAME):
    print(f"ERROR: Cannot process video. File not found: {VIDEO_FILENAME}")
else:
    # Open the video file
    cap = cv2.VideoCapture(VIDEO_FILENAME)
    if not cap.isOpened():
        print(f"Error: Could not open video file {VIDEO_FILENAME}")
    else:
        # Get video properties
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        # Handle potential zero FPS or invalid values
        if fps <= 0:
            print(f"Warning: Video FPS reported as {fps}. Using default 30 FPS for output.")
            fps = 30
        else:
            fps = int(fps)

        # Define the codec and create VideoWriter object
        fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4
        out = cv2.VideoWriter(OUTPUT_VIDEO_FILENAME, fourcc, fps, (frame_width, frame_height))

        print(f"Processing video: {VIDEO_FILENAME} ({frame_width}x{frame_height} @ {fps} FPS)")
        print(f"Output will be saved to: {OUTPUT_VIDEO_FILENAME}")

        frame_count = 0
        start_time = time.time()

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break # End of video

            frame_count += 1
            # Convert frame to tensor (model expects uint8)
            input_tensor = tf.convert_to_tensor(frame)
            input_tensor = input_tensor[tf.newaxis, ...]

            # Run inference
            try:
                detections = detector(input_tensor)
            except Exception as e:
                print(f"Error during inference on frame {frame_count}: {e}")
                continue # Skip this frame

            # Process detections
            num_detections = int(detections.pop('num_detections'))
            detections = {key: value[0, :num_detections].numpy()
                          for key, value in detections.items()}
            detections['num_detections'] = num_detections

            scores = detections['detection_scores']
            boxes = detections['detection_boxes']
            classes = detections['detection_classes']

            for i in range(len(scores)):
                if scores[i] >= CONFIDENCE_THRESHOLD:
                    class_id = int(classes[i])
                    class_name = get_class_name(class_id)

                    if class_name == 'N/A': # Skip if class ID is unknown
                        continue

                    # Bounding box coordinates (normalized [ymin, xmin, ymax, xmax])
                    ymin, xmin, ymax, xmax = boxes[i]
                    (left, right, top, bottom) = (xmin * frame_width, xmax * frame_width,
                                                  ymin * frame_height, ymax * frame_height)
                    left, right, top, bottom = int(left), int(right), int(top), int(bottom)

                    # Check threat status
                    is_threat = class_name.lower() in THREAT_CLASSES

                    # Set color and label
                    color = (0, 0, 255) if is_threat else (0, 255, 0) # Red for threat, Green otherwise
                    label = f"{class_name}: {scores[i]:.2f}"
                    label_size, base_line = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)

                    # Draw bounding box
                    cv2.rectangle(frame, (left, top), (right, bottom), color, 2)

                    # Draw label background and text
                    top_label = max(top, label_size[1]) # Adjust label position if near top edge
                    cv2.rectangle(frame, (left, top_label - label_size[1]), (left + label_size[0], top_label + base_line),
                                  color, cv2.FILLED)
                    cv2.putText(frame, label, (left, top_label), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

                    # Highlight threat
                    if is_threat:
                        threat_text_y = top_label - label_size[1] - 5 # Position above label background
                        threat_text_y = max(threat_text_y, 15) # Ensure it's visible
                        cv2.putText(frame, "THREAT DETECTED", (left, threat_text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

            # Write the processed frame to the output video
            out.write(frame)

            # Optional: Display frame in Colab occasionally or print progress
            if frame_count % 100 == 0:
                print(f"Processed frame {frame_count}...")
                if IN_COLAB:
                    # Display frame in Colab (expects BGR)
                    cv2_imshow(frame)

        # Release resources
        cap.release()
        out.release()
        # cv2.destroyAllWindows() # Only needed if using cv2.imshow locally

        end_time = time.time()
        processing_time = end_time - start_time
        print(f"\nVideo processing complete.")
        print(f"Total frames processed: {frame_count}")
        if processing_time > 0:
            print(f"Total processing time: {processing_time:.2f} seconds")
            print(f"Average FPS: {frame_count / processing_time:.2f}")
        else:
            print("Processing time was negligible or zero frames processed.")
        print(f"Output video saved as: {OUTPUT_VIDEO_FILENAME}")

Processing video: /content/SimplyRuthless.mp4 (720x1280 @ 29 FPS)
Output will be saved to: output_threat_detection.mp4
Error during inference on frame 1: name 'detector' is not defined
Error during inference on frame 2: name 'detector' is not defined
Error during inference on frame 3: name 'detector' is not defined
Error during inference on frame 4: name 'detector' is not defined
Error during inference on frame 5: name 'detector' is not defined
Error during inference on frame 6: name 'detector' is not defined
Error during inference on frame 7: name 'detector' is not defined
Error during inference on frame 8: name 'detector' is not defined
Error during inference on frame 9: name 'detector' is not defined
Error during inference on frame 10: name 'detector' is not defined
Error during inference on frame 11: name 'detector' is not defined
Error during inference on frame 12: name 'detector' is not defined
Error during inference on frame 13: name 'detector' is not defined
Error during infere

## 5. Display Output Video

This section provides a way to display the generated output video directly within the notebook environment (if running in Google Colab).

In [6]:
# Function to display video in Colab
def display_video_colab(video_path):
    if not IN_COLAB:
        print("Video display function is only available in Google Colab.")
        return
    if not os.path.exists(video_path):
        print(f"Video file not found: {video_path}")
        return

    try:
        mp4 = open(video_path, 'rb').read()
        data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
        # Use display() to render HTML object in Colab
        display(HTML(f'''
        <video width=600 controls>
              <source src="{data_url}" type="video/mp4">
        </video>
        '''))
    except NameError: # Handle if IPython/b64encode/display not available
        print("Required libraries for video display (IPython, base64) not available.")
    except Exception as e: # Catch other potential errors
        print(f"Error displaying video: {e}")

# Display the generated video if it exists and in Colab
if os.path.exists(OUTPUT_VIDEO_FILENAME):
    print(f"Attempting to display output video: {OUTPUT_VIDEO_FILENAME}")
    display_video_colab(OUTPUT_VIDEO_FILENAME)
else:
    print(f"Output video file {OUTPUT_VIDEO_FILENAME} not found. Processing might have failed or was skipped.")

Attempting to display output video: output_threat_detection.mp4


## 6. Conclusion and Future Work

This notebook demonstrated a basic pipeline for threat detection using a pre-trained object detection model. Key steps included setting up the environment, loading a model, processing video frames, detecting objects, highlighting potential threats (based on simplified criteria), and generating an annotated output video.

**Potential Future Enhancements:**

*   **Use a model trained specifically for military threats:** The COCO-based model has limitations (e.g., no specific 'rifle' or 'tank' classes). Fine-tuning or using a custom-trained model would improve accuracy for specific threat types.
*   **More sophisticated threat logic:** Implement rule-based systems or sequence analysis to reduce false positives and identify more complex threat scenarios (e.g., a person holding a weapon).
*   **Tracking:** Implement object tracking (e.g., using DeepSORT) to follow detected objects across frames.
*   **Performance Optimization:** Explore model quantization, hardware acceleration (GPU/TPU), or different model architectures for faster inference.
*   **AR Integration:** Adapt the output for rendering on an Augmented Reality device, as mentioned in the initial project description.
*   **Error Handling:** Add more robust error handling for file downloads, model loading, and video processing.